In [26]:
# import packages
import cv2
import glob
import numpy as np
import rawpy
import rawpy.enhance
import gc

In [27]:
# Force OpenCV to run entirely on the CPU and completely disable OpenCL acceleration
# addresses an issue I was having (thanks Gemini)
cv2.ocl.setUseOpenCL(False)

In [28]:
# find all sections in the raw data folder. Assumes folder names are structured as "raw_data/<section_name>"
sampleID = sorted(glob.glob("raw_data/*"))

# remove the leading "raw_data/" from each folder name
sampleID = [folder.replace("raw_data/", "") for folder in sampleID]


# NOTE: Alternatively, you can mannually define a list of core sections to run as below.
sampleID = ["ALHIC2502-102-D"]

print(sampleID)

['ALHIC2502-102-D']


In [29]:
# set params


# set filepaths
raw_data_dir = "raw_data/"
output_dir = "image_outputs/"

# confidence threshold and scaling - note that you might need to tune these to avoid things blowing up!!!
# maybe try starting with 1 and adjusting from there?
scan_conf_thresh_lowres = 1
scan_conf_thresh_highres = 1
scan_min_inclusion_ratio = 0.90
scale_factor = 0.20  # Downsample to 20% size for feature matching


In [30]:

# Loop through folders safely
for sample in sampleID:
    print("\n" + "="*50)
    print(f"Running images from: {sample}")
    print("="*50)

    # 1. Locate files dynamically inside the target sample folder
    nef_paths = sorted(glob.glob(f"{raw_data_dir}/{sample}/*.NEF"))
    print(f" Found {len(nef_paths)} NEF files in folder '{sample}'.")
    
    if len(nef_paths) == 0:
        print("  [Warning] No files found. Skipping this folder.")
        continue

    def count_integrated_images(stitcher):
        try:
            return len(stitcher.cameras())
        except Exception:
            return 0

    attempt_thresholds = [
        (scan_conf_thresh_lowres, scan_conf_thresh_highres),
        (max(0.0, scan_conf_thresh_lowres - 0.15), max(0.0, scan_conf_thresh_highres - 0.10)),
        (max(0.0, scan_conf_thresh_lowres - 0.25), max(0.0, scan_conf_thresh_highres - 0.20)),
    ]

    stitched = False
    for attempt_idx, (lowres_thresh, highres_thresh) in enumerate(attempt_thresholds, start=1):
        print(f" Attempt {attempt_idx}/3 with lowres={lowres_thresh:.2f}, highres={highres_thresh:.2f}")

        # Step 1: Extract low-res thumbnails to calculate alignment geometry
        print(" Step 1: Extracting low-res thumbnails to calculate alignment...")
        low_res_images = []
        for path in nef_paths:
            with rawpy.imread(path) as raw:
                rgb = raw.postprocess(use_camera_wb=True, half_size=True)

            bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

            # Calculate bounds based on target scale factor
            h, w = bgr.shape[:2]
            target_w = int(w * scale_factor * 2)  # Compensates for fast half_size decode
            target_h = int(h * scale_factor * 2)

            small_img = cv2.resize(bgr, (target_w, target_h), interpolation=cv2.INTER_AREA)
            low_res_images.append(small_img)

        # Step 2: Test alignment geometry using flat grid SCANS mode
        print(" Step 2: Aligning 2D zig-zag grid structure...")
        stitcher = cv2.Stitcher_create(cv2.Stitcher_SCANS)
        stitcher.setPanoConfidenceThresh(lowres_thresh)
        status, low_res_scan_preview = stitcher.stitch(low_res_images)
        low_res_integrated = count_integrated_images(stitcher) if status == cv2.Stitcher_OK else 0
        low_res_ratio = low_res_integrated / len(nef_paths)

        # Free up memory allocated to thumbnails immediately
        del low_res_images
        gc.collect()

        if status != cv2.Stitcher_OK:
            print(f"  [Error] Low-res stitching failed with error code: {status}")
            print("  Tip: Verify that the frame-to-frame step overlap is at least 20-30%.")
            if attempt_idx == len(attempt_thresholds):
                print("  Giving up after 3 attempts.")
            else:
                print("  Retrying with lower confidence thresholds...")
            continue

        print(f"  Low-res stitch included {low_res_integrated}/{len(nef_paths)} images ({low_res_ratio:.0%}).")
        if low_res_ratio < scan_min_inclusion_ratio:
            print(f"  [Warning] Low-res inclusion below {scan_min_inclusion_ratio:.0%}; retrying with lower thresholds.")
            if attempt_idx == len(attempt_thresholds):
                print("  Giving up after 3 attempts.")
            continue

        print("  Grid layout successfully calculated!")

        # Step 3: High resolution processing with safe RAM footprints
        print(" Step 3: Generating final high-res grid composition...")
        stitcher_high = cv2.Stitcher_create(cv2.Stitcher_SCANS)
        stitcher_high.setRegistrationResol(0.6)
        stitcher_high.setPanoConfidenceThresh(highres_thresh)
        high_res_images = []
        for path in nef_paths:
            with rawpy.imread(path) as raw:
                # Full quality decode
                rgb = raw.postprocess(use_camera_wb=True, half_size=False)
            bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
            high_res_images.append(bgr)
        print("  Assembling canvas matrices...")
        status, high_res_scan_result = stitcher_high.stitch(high_res_images)
        high_res_integrated = count_integrated_images(stitcher_high) if status == cv2.Stitcher_OK else 0
        high_res_ratio = high_res_integrated / len(nef_paths)

        # Instantly wipe the high-res list elements out of memory cache
        del high_res_images
        gc.collect()

        if status == cv2.Stitcher_OK and high_res_ratio >= scan_min_inclusion_ratio:
            print(f"  High-res stitch included {high_res_integrated}/{len(nef_paths)} images ({high_res_ratio:.0%}).")
            # Standardized file pathing structure
            output_path = f"{output_dir}/{sample}_stitched_output.jpg"
            cv2.imwrite(output_path, high_res_scan_result)
            print(f"  Success! Stitched grid saved to: {output_path}")

            # Completely drop panorama matrix representation to reset memory baseline
            del high_res_scan_result
            gc.collect()
            stitched = True
            break

        print(f"  [Warning] High-res stitch only included {high_res_integrated}/{len(nef_paths)} images ({high_res_ratio:.0%}).")
        print("  Retrying with lower confidence thresholds...")
        if status == cv2.Stitcher_OK:
            del high_res_scan_result
            gc.collect()

    if not stitched:
        print("  [Error] Could not achieve a stitch that included at least 90% of the sub-images after 3 attempts.")



Running images from: ALHIC2502-102-D
 Found 57 NEF files in folder 'ALHIC2502-102-D'.
 Attempt 1/3 with lowres=1.00, highres=1.00
 Step 1: Extracting low-res thumbnails to calculate alignment...
 Step 2: Aligning 2D zig-zag grid structure...
  Low-res stitch included 32/57 images (56%).
  [Warning] Low-res inclusion below 90%; retrying with lower thresholds.
 Attempt 2/3 with lowres=0.85, highres=0.90
 Step 1: Extracting low-res thumbnails to calculate alignment...
 Step 2: Aligning 2D zig-zag grid structure...
  Low-res stitch included 32/57 images (56%).
  [Warning] Low-res inclusion below 90%; retrying with lower thresholds.
 Attempt 3/3 with lowres=0.75, highres=0.80
 Step 1: Extracting low-res thumbnails to calculate alignment...
 Step 2: Aligning 2D zig-zag grid structure...
  Low-res stitch included 57/57 images (100%).
  Grid layout successfully calculated!
 Step 3: Generating final high-res grid composition...
  Assembling canvas matrices...
  High-res stitch included 57/57 i